# InClassActivity2 — SRILM

Roll No: M25AI2151

I first set up and compiled SRILM in Google Colab. Since the build took a long time, I saved the compiled installation to Google Drive and reused it when Colab disconnected or restarted. The later cells continue from that work and run the language-model experiments. If the saved build is missing, the setup section explains how to rebuild it.

## 1. Initial setup and SRILM compilation

This is the original setup stage of the work. It is kept separate from the later restore step: compile SRILM only if the saved build is not available. Once compilation succeeds, save the installation archive in Drive so it can be reused after a runtime reset.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import shutil
import tarfile
import subprocess
from pathlib import Path

workdir = Path("/content/drive/MyDrive/InClassActivity2")
workdir.mkdir(parents=True, exist_ok=True)

archive = workdir / "srilm-built.tar.gz"
source_dir = Path("/content/srilm")
bin_dir = Path("/content/srilm/bin/i686-m64")

# Reuse the saved build when available. Otherwise compile SRILM once and archive it.
if (bin_dir / "ngram-count").is_file() and (bin_dir / "ngram").is_file():
    print("SRILM is already available in this runtime.")
elif archive.is_file():
    print("Restoring the previously compiled SRILM build from Drive...")
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall("/content")
else:
    print("No saved build found. Cloning and compiling SRILM for the first time...")
    if source_dir.exists():
        shutil.rmtree(source_dir)
    subprocess.run(
        ["git", "clone", "https://github.com/BitSpeech/SRILM.git", str(source_dir)],
        check=True
    )
    makefile = source_dir / "Makefile"
    if makefile.exists():
        content = makefile.read_text()
        content = content.replace(
            "# SRILM = /home/speech/stolcke/project/srilm/devel",
            "SRILM = /content/srilm"
        )
        makefile.write_text(content)
    subprocess.run(["make", "World"], cwd=str(source_dir), check=True)

    print("Compilation finished. Saving the build archive to Drive...")
    with tarfile.open(archive, "w:gz") as tar:
        tar.add(source_dir, arcname="srilm")

if not ((bin_dir / "ngram-count").is_file() and (bin_dir / "ngram").is_file()):
    raise FileNotFoundError(
        "SRILM binaries were not found after restore/build. Check the archive structure or compilation output."
    )

os.environ["PATH"] = str(bin_dir) + ":" + os.environ["PATH"]
print("SRILM ready:", (bin_dir / "ngram-count").is_file() and (bin_dir / "ngram").is_file())
!ngram-count -version
!ngram -version

# Restore corpus files saved during the earlier work.
for name in ["brown-train.txt", "brown-dev.txt", "brown-test.txt"]:
    src = workdir / name
    dst = Path("/content") / name
    if not dst.is_file():
        if not src.is_file():
            raise FileNotFoundError(f"Corpus file not found in Drive: {src}")
        shutil.copy2(src, dst)
    print(f"{name}: {dst.stat().st_size:,} bytes")


## 2. Prepare training text and build/evaluate language models

Sentence-boundary markers are added before counting. The main experiment cell builds the bigram count file, unsmoothed bigram, Kneser–Ney bigram and trigram, and unigram/bigram models used for interpolation. It saves commands and outputs in the Drive folder.

In [ ]:

import os
import re
import shutil
import subprocess
from pathlib import Path

workdir = Path("/content/drive/MyDrive/InClassActivity2")
workdir.mkdir(parents=True, exist_ok=True)

# Confirm SRILM and restore inputs if needed
bin_dir = "/content/srilm/bin/i686-m64"
os.environ["PATH"] = bin_dir + ":" + os.environ["PATH"]

for name in ["brown-train.txt", "brown-dev.txt", "brown-test.txt"]:
    src = workdir / name
    dst = Path("/content") / name
    if not dst.exists():
        if not src.exists():
            raise FileNotFoundError(f"Missing corpus: {src}")
        shutil.copy2(src, dst)

# Add sentence-boundary markers to training sentences
train_marked = Path("/content/brown-train-marked.txt")
with open("/content/brown-train.txt", encoding="utf-8",
          errors="replace") as fin, open(train_marked, "w", encoding="utf-8") as fout:
    for line in fin:
        words = line.strip().split()
        if words:
            fout.write("<s> " + " ".join(words) + " </s>\n")

# Persist prepared training text
shutil.copy2(train_marked, workdir / train_marked.name)

log_path = workdir / "experiment-results.txt"
commands_path = workdir / "experiment-commands.txt"

def run_cmd(args, label, log_file):
    log_file.write(f"\n\n===== {label} =====\n")
    log_file.write("$ " + " ".join(map(str, args)) + "\n")
    log_file.flush()
    result = subprocess.run(
        list(map(str, args)), text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    log_file.write(result.stdout)
    log_file.flush()
    print(f"{label}: {'OK' if result.returncode == 0 else 'FAILED'}")
    if result.returncode:
        print(result.stdout[-2500:])
        raise RuntimeError(f"{label} failed")
    return result.stdout

train = str(train_marked)
test = "/content/brown-test.txt"
dev = "/content/brown-dev.txt"

experiments = [
    ("Bigram counts",
     ["ngram-count", "-text", train, "-order", "2",
      "-write", "/content/brown-train.count"]),
    ("Unsmoothed bigram",
     ["ngram-count", "-read", "/content/brown-train.count",
      "-order", "2", "-lm", "/content/brown-bigram-unsmoothed.lm",
      "-addsmooth", "0", "-unk"]),
    ("Kneser-Ney bigram",
     ["ngram-count", "-text", train, "-order", "2",
      "-lm", "/content/brown-bigram-kn.lm", "-unk", "-kndiscount"]),
    ("Kneser-Ney trigram",
     ["ngram-count", "-text", train, "-order", "3",
      "-lm", "/content/brown-trigram-kn.lm", "-unk", "-kndiscount"]),
    ("Unigram model for interpolation",
     ["ngram-count", "-text", train, "-order", "1",
      "-lm", "/content/brown-unigram.lm", "-unk"]),
    ("Bigram model for interpolation",
     ["ngram-count", "-text", train, "-order", "2",
      "-lm", "/content/brown-bigram-interp.lm", "-unk",
      "-kndiscount"])
]

with open(log_path, "w", encoding="utf-8") as log, \
     open(commands_path, "w", encoding="utf-8") as cmds:
    for label, args in experiments:
        cmds.write(label + ": " + " ".join(args) + "\n")
        run_cmd(args, label, log)

    # Evaluate models with unknown-word handling enabled consistently
    for label, model in [
        ("Unsmoothed bigram test evaluation", "/content/brown-bigram-unsmoothed.lm"),
        ("Kneser-Ney bigram test evaluation", "/content/brown-bigram-kn.lm"),
        ("Kneser-Ney trigram test evaluation", "/content/brown-trigram-kn.lm")
    ]:
        run_cmd(["ngram", "-lm", model, "-ppl", test, "-unk"], label, log)

# Copy model/count artifacts to Drive
for name in [
    "brown-train.count",
    "brown-bigram-unsmoothed.lm",
    "brown-bigram-kn.lm",
    "brown-trigram-kn.lm",
    "brown-unigram.lm",
    "brown-bigram-interp.lm"
]:
    src = Path("/content") / name
    if src.exists():
        shutil.copy2(src, workdir / name)

print("\nSaved logs:", log_path)
print("Saved commands:", commands_path)
print("Saved models and count file in:", workdir)


Bigram counts: OK
Unsmoothed bigram: OK
Kneser-Ney bigram: OK
Kneser-Ney trigram: OK
Unigram model for interpolation: OK
Bigram model for interpolation: OK
Unsmoothed bigram test evaluation: OK
Kneser-Ney bigram test evaluation: OK
Kneser-Ney trigram test evaluation: OK

Saved logs: /content/drive/MyDrive/InClassActivity2/experiment-results.txt
Saved commands: /content/drive/MyDrive/InClassActivity2/experiment-commands.txt
Saved models and count file in: /content/drive/MyDrive/InClassActivity2


## 3. Check unknown-word handling

Evaluate the unsmoothed bigram model without and with `-unk` to see how unknown words affect the test-set output.

In [ ]:
import os
os.environ["PATH"] = "/content/srilm/bin/i686-m64:" + os.environ["PATH"]

print("Without -unk:")
!ngram -lm /content/brown-bigram-unsmoothed.lm -ppl /content/brown-test.txt

print("\nWith -unk:")
!ngram -lm /content/brown-bigram-unsmoothed.lm -ppl /content/brown-test.txt -unk


## 4. Preprocessing: punctuation and stopword removal

This cell creates the punctuation-removed and punctuation-removed-plus-stopword-removed training/test files, trains bigram models, and saves their evaluation outputs.

In [ ]:

import os
import re
import shutil
import subprocess
from pathlib import Path

workdir = Path("/content/drive/MyDrive/InClassActivity2")
os.environ["PATH"] = "/content/srilm/bin/i686-m64:" + os.environ["PATH"]

# English stopwords
try:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    stopwords = set(ENGLISH_STOP_WORDS)
except ImportError:
    stopwords = {
        "a", "an", "the", "and", "or", "but", "if", "in", "on", "at",
        "to", "of", "for", "with", "is", "are", "was", "were", "be",
        "been", "being", "it", "its", "this", "that", "these", "those",
        "i", "you", "he", "she", "we", "they", "my", "your", "his",
        "her", "our", "their", "as", "by", "from", "not", "have", "has"
    }

def preprocess(source, destination, remove_stopwords=False):
    with open(source, encoding="utf-8", errors="replace") as fin, \
         open(destination, "w", encoding="utf-8") as fout:
        for line in fin:
            # Remove punctuation, preserving letters, digits and whitespace
            cleaned = re.sub(r"[^\w\s]", "", line)
            words = cleaned.lower().split()
            if remove_stopwords:
                words = [w for w in words if w not in stopwords]
            if words:
                fout.write("<s> " + " ".join(words) + " </s>\n")

datasets = [
    ("nopunct", False),
    ("nopunct_nostop", True)
]

for tag, remove_sw in datasets:
    for split in ["train", "test"]:
        source = f"/content/brown-{split}.txt"
        destination = f"/content/brown-{split}-{tag}.txt"
        preprocess(source, destination, remove_sw)
        shutil.copy2(destination, workdir / Path(destination).name)

results = workdir / "preprocessing-results.txt"

with open(results, "w", encoding="utf-8") as log:
    for tag, _ in datasets:
        train = f"/content/brown-train-{tag}.txt"
        test = f"/content/brown-test-{tag}.txt"
        model = f"/content/brown-{tag}-bigram.lm"

        build = subprocess.run(
            ["ngram-count", "-text", train, "-order", "2",
             "-lm", model, "-unk", "-kndiscount"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
        )
        log.write(f"\n===== {tag}: model build =====\n{build.stdout}\n")
        if build.returncode:
            raise RuntimeError(f"Model build failed for {tag}")

        evaluation = subprocess.run(
            ["ngram", "-lm", model, "-ppl", test, "-unk"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
        )
        log.write(f"===== {tag}: test evaluation =====\n{evaluation.stdout}\n")
        if evaluation.returncode:
            raise RuntimeError(f"Evaluation failed for {tag}")

        shutil.copy2(model, workdir / Path(model).name)
        print(f"{tag}: completed")

print("Results saved to:", results)


nopunct: completed
nopunct_nostop: completed
Results saved to: /content/drive/MyDrive/InClassActivity2/preprocessing-results.txt


## 5. Interpolation tuning on the development set

First, the unigram/bigram mixture weight is varied while evaluating on the development set. The selected weight is then evaluated once on the test set.

In [ ]:

import os
import re
import subprocess
import shutil
from pathlib import Path

workdir = Path("/content/drive/MyDrive/InClassActivity2")
os.environ["PATH"] = "/content/srilm/bin/i686-m64:" + os.environ["PATH"]

unigram = "/content/brown-unigram.lm"
bigram = "/content/brown-bigram-interp.lm"
dev = "/content/brown-dev.txt"
test = "/content/brown-test.txt"

for path in [unigram, bigram, dev, test]:
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Required file missing: {path}")

weights = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
results = []
log_path = workdir / "interpolation-results.txt"

with open(log_path, "w", encoding="utf-8") as log:
    log.write("Unigram + bigram interpolation; tuning on development set\n")

    for lam in weights:
        cmd = [
            "ngram", "-lm", unigram,
            "-mix-lm", bigram,
            "-lambda", str(lam),
            "-ppl", dev, "-unk"
        ]
        result = subprocess.run(
            cmd, capture_output=True, text=True
        )
        output = result.stdout + "\n" + result.stderr
        log.write(f"\n===== lambda={lam} on development set =====\n")
        log.write(" ".join(cmd) + "\n" + output + "\n")

        if result.returncode != 0:
            raise RuntimeError(f"Interpolation failed for lambda={lam}:\n{output}")

        match = re.search(r"ppl1=\s*([0-9.]+)", output)
        if not match:
            raise RuntimeError(f"Could not parse ppl1 for lambda={lam}:\n{output}")

        ppl1 = float(match.group(1))
        results.append((lam, ppl1))
        print(f"lambda={lam:.1f} | development ppl1={ppl1:.4f}")

best_lambda, best_dev_ppl1 = min(results, key=lambda x: x[1])
print(f"\nSelected lambda: {best_lambda}")
print(f"Best development ppl1: {best_dev_ppl1:.4f}")

# Evaluate the selected weight ONCE on the test set
cmd = [
    "ngram", "-lm", unigram,
    "-mix-lm", bigram,
    "-lambda", str(best_lambda),
    "-ppl", test, "-unk"
]
result = subprocess.run(cmd, capture_output=True, text=True)
test_output = result.stdout + "\n" + result.stderr

if result.returncode != 0:
    raise RuntimeError(test_output)

with open(log_path, "a", encoding="utf-8") as log:
    log.write("\n===== Selected interpolation on test set =====\n")
    log.write(f"Selected lambda: {best_lambda}\n")
    log.write(" ".join(cmd) + "\n" + test_output)

print("\nTEST-SET RESULTS")
print(test_output)
print("\nFull results saved to:", log_path)


lambda=0.1 | development ppl1=427.8440
lambda=0.2 | development ppl1=434.6117
lambda=0.3 | development ppl1=446.6367
lambda=0.4 | development ppl1=464.2692
lambda=0.5 | development ppl1=488.8415
lambda=0.6 | development ppl1=522.9922
lambda=0.7 | development ppl1=571.9004
lambda=0.8 | development ppl1=647.0939
lambda=0.9 | development ppl1=782.7431

Selected lambda: 0.1
Best development ppl1: 427.8440

TEST-SET RESULTS
file /content/brown-test.txt: 14334 sentences, 305056 words, 0 OOVs
0 zeroprobs, logprob= -812593.4 ppl= 350.1098 ppl1= 461.0538



Full results saved to: /content/drive/MyDrive/InClassActivity2/interpolation-results.txt


## 6. Tune interpolation weight and add-smoothing value

This grid search compares the listed `-addsmooth` values and mixture weights on the development set. The best tested configuration is then evaluated on the test set. The best setting is the best among this grid, not necessarily the global optimum.

In [ ]:

import os
import re
import subprocess
from pathlib import Path

os.environ["PATH"] = "/content/srilm/bin/i686-m64:" + os.environ["PATH"]
workdir = Path("/content/drive/MyDrive/InClassActivity2")

train = "/content/brown-train-marked.txt"
dev = "/content/brown-dev.txt"
test = "/content/brown-test.txt"

# Verify the prepared corpus exists
assert os.path.isfile(train), "Training file missing"
assert os.path.isfile(dev), "Development file missing"
assert os.path.isfile(test), "Test file missing"

# Explore several small add-smoothing values and mixture weights.
# Save all development results before choosing the best configuration.
smoothing_values = [0.01, 0.1, 0.5, 1.0]
weights = [0.1, 0.3, 0.5, 0.7, 0.9]
results = []
log_path = workdir / "interpolation-grid-results.txt"

with open(log_path, "w", encoding="utf-8") as log:
    for smooth in smoothing_values:
        uni = f"/content/uni_add_{smooth}.lm"
        bi = f"/content/bi_add_{smooth}.lm"

        for order, model in [(1, uni), (2, bi)]:
            build = subprocess.run(
                ["ngram-count", "-text", train, "-order", str(order),
                 "-lm", model, "-addsmooth", str(smooth), "-unk"],
                capture_output=True, text=True
            )
            if build.returncode:
                raise RuntimeError(build.stdout + build.stderr)

        for lam in weights:
            cmd = [
                "ngram", "-lm", uni, "-mix-lm", bi,
                "-lambda", str(lam), "-ppl", dev, "-unk"
            ]
            run = subprocess.run(cmd, capture_output=True, text=True)
            output = run.stdout + run.stderr

            if run.returncode:
                raise RuntimeError(output)

            match = re.search(r"ppl1=\s*([0-9.]+)", output)
            if not match:
                raise RuntimeError("Could not parse development ppl1:\n" + output)

            ppl1 = float(match.group(1))
            results.append((ppl1, smooth, lam))
            log.write(
                f"addsmooth={smooth}, lambda={lam}, "
                f"development_ppl1={ppl1}\n{output}\n"
            )
            print(f"addsmooth={smooth:<4} lambda={lam:.1f} dev_ppl1={ppl1:.4f}")

best_ppl1, best_smooth, best_lambda = min(results)

print("\nBest development configuration:")
print("addsmooth =", best_smooth)
print("lambda =", best_lambda)
print("development ppl1 =", best_ppl1)

# Rebuild the selected models and evaluate on the held-out test set.
uni = "/content/selected-unigram.lm"
bi = "/content/selected-bigram.lm"

for order, model in [(1, uni), (2, bi)]:
    build = subprocess.run(
        ["ngram-count", "-text", train, "-order", str(order),
         "-lm", model, "-addsmooth", str(best_smooth), "-unk"],
        capture_output=True, text=True
    )
    if build.returncode:
        raise RuntimeError(build.stdout + build.stderr)

cmd = [
    "ngram", "-lm", uni, "-mix-lm", bi,
    "-lambda", str(best_lambda), "-ppl", test, "-unk"
]
run = subprocess.run(cmd, capture_output=True, text=True)
if run.returncode:
    raise RuntimeError(run.stdout + run.stderr)

test_output = run.stdout + run.stderr
print("\nSelected configuration on test set:")
print(test_output)

with open(log_path, "a", encoding="utf-8") as log:
    log.write(
        f"\nSELECTED: addsmooth={best_smooth}, lambda={best_lambda}\n"
        f"Development ppl1={best_ppl1}\n"
        f"Test command: {' '.join(cmd)}\n{test_output}\n"
    )

print("Saved results to:", log_path)


addsmooth=0.01 lambda=0.1 dev_ppl1=909.3159
addsmooth=0.01 lambda=0.3 dev_ppl1=879.5534
addsmooth=0.01 lambda=0.5 dev_ppl1=913.8142
addsmooth=0.01 lambda=0.7 dev_ppl1=1010.1660
addsmooth=0.01 lambda=0.9 dev_ppl1=1254.9320
addsmooth=0.1  lambda=0.1 dev_ppl1=1228.0970
addsmooth=0.1  lambda=0.3 dev_ppl1=1136.0690
addsmooth=0.1  lambda=0.5 dev_ppl1=1139.3710
addsmooth=0.1  lambda=0.7 dev_ppl1=1208.4570
addsmooth=0.1  lambda=0.9 dev_ppl1=1399.7910
addsmooth=0.5  lambda=0.1 dev_ppl1=1725.1670
addsmooth=0.5  lambda=0.3 dev_ppl1=1483.4230
addsmooth=0.5  lambda=0.5 dev_ppl1=1420.9910
addsmooth=0.5  lambda=0.7 dev_ppl1=1437.4650
addsmooth=0.5  lambda=0.9 dev_ppl1=1550.3190
addsmooth=1.0  lambda=0.1 dev_ppl1=2031.3330
addsmooth=1.0  lambda=0.3 dev_ppl1=1677.8420
addsmooth=1.0  lambda=0.5 dev_ppl1=1565.6360
addsmooth=1.0  lambda=0.7 dev_ppl1=1543.6700
addsmooth=1.0  lambda=0.9 dev_ppl1=1607.2580

Best development configuration:
addsmooth = 0.01
lambda = 0.3
development ppl1 = 879.5534

Selected co

## 7. Check saved experiment files

Use this cell to confirm that the logs and model files are present in Drive.

In [ ]:
import os

workdir = "/content/drive/MyDrive/InClassActivity2"
for name in sorted(os.listdir(workdir)):
    path = os.path.join(workdir, name)
    if os.path.isfile(path):
        print(f"{name} | {os.path.getsize(path):,} bytes")
